<a href="https://colab.research.google.com/github/ksuplee/tensorflow-nlp-tutorial/blob/main/12_HuggingFace/12_01_chat_demo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 1. Hugging Face 추론 API 사용

## 1.1 추론 API 사용(OpenAI 방식과 가장 유사)

Hugging Face의 InferenceClient는 OpenAI의 API 구조와 거의 동일하게 설계되어 있어 코드 변경을 최소화할 수 있습니다. 무거운 모델을 직접 다운로드할 필요가 없습니다.

사전 준비:

- Hugging Face 개념 이해: https://wikidocs.net/296874  

- Hugging Face API Key 생성: https://wikidocs.net/328722  

- Colab 왼쪽 🔑(보안 비밀) 탭에 HF_TOKEN이라는 이름으로 Hugging Face Access Token을 저장해야 합니다.  

- !pip install huggingface_hub 실행 필요  

In [ ]:
!pip install huggingface_hub

In [ ]:
from huggingface_hub import InferenceClient
from google.colab import userdata

# Colab 시크릿에서 Hugging Face 토큰 불러오기
HF_TOKEN = userdata.get('HF_TOKEN')

# OpenAI Client와 동일한 구조의 InferenceClient 초기화
# 한국어 성능이 좋은 오픈소스 모델(예: Qwen2.5 또는 Gemma2) 지정
# Qwen/Qwen2.5-72B-Instruct는 Alibaba의 Qwen 팀이 공개한 약 727억 개 파라미터의
#   명령어 수행형 대규모 언어모델입니다. 질문답변, 문서 요약, 번역, 코드 작성,
#   수학 문제, JSON 생성, 대화 등에 사용할 수 있음
client = InferenceClient(
    model="Qwen/Qwen2.5-72B-Instruct",
    token=HF_TOKEN
)

resp = client.chat.completions.create(
    messages=[
        {"role": "system", "content": "너는 유능한 AI 어시스턴트다."},
        {"role": "user",   "content": "트랜스포머의 핵심 아이디어를 한 문장으로 설명해줘."},
        {"role": "user",   "content": "창발 현상에 대해 설명해줘."},
    ],
    temperature=0.3,
    max_tokens=100,
)

print(resp.choices[0].message.content)

트랜스포머의 핵심 아이디어는 self-attention 메커니즘을 통해 입력 시퀀스의 모든 단어 간의 관계를 평행하게 계산하는 것입니다.

창발 현상(Emergent Phenomenon)은 복잡한 시스템에서 개별 구성 요소들이 상호작용하면서 예상치 못한 새로운 특성이나 행동이 나타나는 현상을 말합니다. 이는 인공지


## 1.2 Transformers 라이브러리를 통한 로컬 GPU 구동

외부 서버에 의존하지 않고 Colab의 GPU 자원을 직접 사용하여 모델을 돌리고 싶을 때 사용하는 방식입니다.

사전 준비:

- 런타임 유형을 GPU로 변경해야 합니다.  

- !pip install transformers accelerate 실행 필요  

In [ ]:
!pip install transformers accelerate

In [ ]:
import torch
from transformers import pipeline
from google.colab import userdata

# 제한된(Gated) 모델을 사용할 경우 토큰 필요
HF_TOKEN = userdata.get('HF_TOKEN')

# 텍스트 생성 파이프라인 구축 (Colab 무료 GPU에 맞는 7B~9B 사이즈 권장)
pipe = pipeline(
    "text-generation",
    model="TinyLlama/TinyLlama-1.1B-Chat-v1.0", # 접근 가능한 공개 모델로 변경
    token=HF_TOKEN,
    device_map="auto",
    torch_dtype=torch.float16 # 메모리 절약을 위해 16bit 정밀도 사용
)

messages = [
    {"role": "system", "content": "너는 유능한 AI 어시스턴트다."},
    {"role": "user",   "content": "트랜스포머의 핵심 아이디어를 한 문장으로 설명해줘."},
    {"role": "user",   "content": "디코더 기반 자기회귀 생성을 설명해줘."},
]

# 파이프라인에 메시지 전달
result = pipe(
    messages,
    max_new_tokens=100,
    temperature=0.3,
    do_sample=True # temperature를 적용하기 위해 필요
)

# 생성된 답변의 텍스트만 추출
print(result[0]['generated_text'][-1]['content'])

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

[transformers] Both `max_new_tokens` (=100) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


디코더 기반 자기회귀 생성 (Deep Recurrent Neural Networks, DRNN)는 네트워크 구조를 잘 알고 있는 딥러닝 모델을 사용하는 것입니다.


# 2. Hugging Face 모델 활용

오픈웨이트 모델은 transformers 라이브러리의 pipeline API로 직접 사용할 수 있다.  

| 방식 | 장점 | 단점 |
|------|------|------|
| **API 호출(폐쇄형)** | 최고 성능, 인프라 불필요 | 비용 발생, 의존성 |
| **직접 구동(오픈웨이트)** | 무료, 커스터마이징 가능 | GPU 필요, 성능 한계 |

## 2.1 간단한 문장 이어쓰기 모델 적용 (결과 불안정)

In [11]:
from transformers import pipeline

generator = pipeline("text-generation", model="skt/ko-gpt-trinity-1.2B-v0.5")
result = generator("자연어처리란", max_new_tokens=100, temperature=0.6)
print("\n=== Output ===\n")
print(result[0]["generated_text"])

Loading weights:   0%|          | 0/292 [00:00<?, ?it/s]

[transformers] GPT2LMHeadModel LOAD REPORT from: skt/ko-gpt-trinity-1.2B-v0.5
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
transformer.h.{0...23}.attn.masked_bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
[transformers] Both `max_new_tokens` (=100) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



=== Output ===

자연어처리란#104EBF▁</span>▁(<span▁style="color:#104EB7;">T</span>▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁


## 2.2 문장 생성 라이브러리 업데이트  

Hugging Face의 text-generation 파이프라인은 Instruct 모델에 role과 content로 구성된 대화 형식을 전달하면 모델의 채팅 템플릿을 자동 적용합니다.  

반환 결과에서는 마지막 assistant 메시지의 content를 꺼내면 생성 답변만 출력할 수 있습니다.  

In [ ]:
!pip install -U transformers accelerate torch

In [10]:
from transformers import pipeline

model_id = "Qwen/Qwen2.5-1.5B-Instruct"

generator = pipeline(
    task="text-generation",
    model=model_id,
    device_map="auto"
)

messages = [
    {
        "role": "system",
        "content": "당신은 자연어처리를 쉽게 설명하는 한국어 강사입니다."
    },
    {
        "role": "user",
        "content": "자연어처리란 무엇인지 세 문장으로 설명해 주세요."
    }
]

result = generator(
    messages,
    max_new_tokens=256,
    do_sample=True,
    temperature=0.9,
    top_p=0.9,
    repetition_penalty=1.1
)

answer = result[0]["generated_text"][-1]["content"]
print("\n=== Output ===\n")
print(answer)

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



=== Output ===

1. **문장 1:** "자연어처리는 컴퓨터가 인간의 언어를 이해하고 사용할 수 있도록 하는 기술과 알고리즘을 사용하여 수행되는 분야입니다."
   
2. **문장 2:** "대상 언어는 다양한 언어로, 하지만 이론적으로는 모든 언어(일본어, 영어 등)에 대한 처리 가능합니다."

3. **문장 3:** "이러한 처리는 인공지능의 하나로, 문서 검색, 음성 인식, 번역, 메시지를 이해하는 등의 실제 프로젝트에서 활용됩니다."


# 3. 생성 파라미터 조절과 출력 비교

같은 프롬프트(prompt, 모델에 보내는 입력 텍스트)에 파라미터를 달리하여 출력을 비교한다.  

| 파라미터 | 설명 | 낮은 값 | 높은 값 |
|----------|------|---------|---------|
| `temperature` | softmax 출력 분포의 날카로움 조절 | 결정적·일관적 | 다양·창의적·불안정 |
| `top_p` | 누적 확률 상위 토큰만 샘플링 (Nucleus Sampling) | 보수적(적은 후보) | 다양(많은 후보) |
| `max_tokens` | 생성할 최대 토큰 수 | 짧은 출력 | 긴 출력 허용 |

In [7]:
# !pip install huggingface_hub 실행 필요
from huggingface_hub import InferenceClient
from google.colab import userdata

# 1. Colab 보안 비밀(Secrets)에서 토큰 불러오기
HF_TOKEN = userdata.get('HF_TOKEN')

# 2. OpenAI 클라이언트를 Hugging Face InferenceClient로 대체
client = InferenceClient(api_key=HF_TOKEN)

def generate(prompt, temp=0.0, top_p=1.0):
    # Hugging Face API는 temperature=0을 허용하지 않으므로, 0일 경우 매우 작은 값으로 대체
    if temp == 0.0:
        temp = 0.01

    r = client.chat.completions.create(
        model="Qwen/Qwen2.5-72B-Instruct", # 한국어 성능이 우수한 오픈소스 모델
        messages=[{"role": "user", "content": prompt}],
        temperature=temp,
        top_p=top_p,
        max_tokens=80
    )
    return r.choices[0].message.content

prompt = "인공지능의 미래를 한 문단으로 서술하라."

print("=== temperature=0 (결정적) ===")
print(generate(prompt, temp=0.0))

print("\n=== temperature=0.7 (적당한 다양성) ===")
print(generate(prompt, temp=0.7))

print("\n=== temperature=1.2 (매우 창의적) ===")
print(generate(prompt, temp=1.2))


=== temperature=0 (결정적) ===
인공지능의 미래는 빠르게 진화하며 다양한 산업과 일상생활에 깊이 스며들 것입니다. AI는 더욱 정교해진 자연어 처리와 감성 인식 능력을 바탕으로 인간과의 상호작용을 더욱 자연스럽고 효과적으로 만들 것이며, 의료, 교육, 금

=== temperature=0.7 (적당한 다양성) ===
인공지능의 미래는 빠르게 진화하며, 다양한 산업 분야에서 혁신을 이끌 것으로 전망됩니다. 더 발전된 AI 기술은 보다 정교하고 인간 중심적인 서비스를 제공하며, 의료, 교육, 금융 등 다양한 분야에서 개인화되고 효율적인 솔루션을

=== temperature=1.2 (매우 창의적) ===
인공지능의 미래는 무궁무진한 가능성을 지니고 있으며, 기술 발전에 따라 더욱 첨단化되고 인간의 일상생활과 사회 전반에 깊이 융합될 것으로 전망됩니다. 하지만 이로 인한 윤리적 문제와 고용주시قم 등 다양한 과제도 함께 부각되고 있어,
